In [1]:
import json

In [2]:
# Load the data
def load_data(filename):
    with open(filename, "r") as f:
        data = json.load(f)
    return data

In [3]:
data = load_data("store_data.json")

In [4]:
 # Clean & structure the data
def clean_data(data):
    text_to_num = {"one":1.0, "two":2.0, "three":3.0, "four":4.0, "five":5.0}
    cleaned_data = []
    unique_users = set() 
    
    for user in data:
        #Clean ratings
        raw_rating = user.get("rating","") #Give the value, and incase key doesn't exist, the gives ""
        if isinstance(raw_rating, str): #Avoids Integers, float, None
            raw_rating = raw_rating.strip().lower()
        if raw_rating in text_to_num: #four -> 4.0
            raw_rating = text_to_num[raw_rating]
        else:
            try:
                raw_rating = float(raw_rating)
            except (TypeError, ValueError):
                raw_rating = None
        user["rating"] = raw_rating
        
        #Handling missing age
        raw_age = user.get("age",None)
        if isinstance(raw_age, str):
            raw_age = raw_age.strip()
        try:
            raw_age = int(raw_age)
        except (TypeError, ValueError):
            raw_age = None
        user["age"] = raw_age

        #Handling Duplicate Users
        raw_name = user.get("name","")
        if isinstance(raw_name, str):
            raw_name = raw_name.strip().lower()
        if raw_name:
            if raw_name not in unique_users:
                unique_users.add(raw_name)
                cleaned_data.append(user)
        else: #To add names like empty, or None
            cleaned_data.append(user)
        
    return cleaned_data

In [5]:
data = clean_data(data)
for user in data:
    print(user)

{'name': 'Alice', 'rating': 5.0, 'feedback': 'Great product!!', 'age': 25}
{'name': 'Bob', 'rating': 4.0, 'feedback': 'ok but late Delivery', 'age': 30}
{'name': ' Charlie', 'rating': 2.0, 'feedback': 'BAD EXPERIENCE ', 'age': None}
{'name': 'Diana', 'feedback': 'Loved it!', 'rating': 5.0, 'age': 28}
{'name': 'Eve', 'rating': 3.5, 'feedback': 'Average - could be better', 'age': 20}


In [6]:
def dump_data(filename, data):
    with open(filename, "w") as f:
        json.dump(data, f)

In [7]:
dump_data("store_data.json", data)

In [8]:
#Getting meaningful insights!
def get_insights(data):
    total_rating = 0
    poor_rating = 0
    for user in data:
        if user["rating"]:
            total_rating+=user["rating"]
            if user["rating"] < 3:
                poor_rating+=1

    #Average rating
    avg_rating = total_rating/len(data)
    print("Average Rating =", avg_rating)

    #Percentage of poor rating
    poor_rating = (poor_rating/len(data))*100
    print(f"{poor_rating}% of users gave rating < 3")    

In [9]:
get_insights(data)

Average Rating = 3.9
20.0% of users gave rating < 3


In [10]:
#Building recommendation feature
def get_recommendations(data):
    recommendations = []

    for user in data:
        curr_recomm = {}
        curr_recomm["name"] = user["name"]
        
        if user["rating"]>=4: #Recommend products of the same brand, i.e. Apple
            curr_recomm["brand"] = "Apple"
        else:
            curr_recomm["brand"] = "Samsung"
            
        recommendations.append(curr_recomm) 
    return recommendations

In [11]:
get_recommendations(data)

[{'name': 'Alice', 'brand': 'Apple'},
 {'name': 'Bob', 'brand': 'Apple'},
 {'name': ' Charlie', 'brand': 'Samsung'},
 {'name': 'Diana', 'brand': 'Apple'},
 {'name': 'Eve', 'brand': 'Samsung'}]